### Extracting English Text from the Canadian Bills
Use the ``` scripts/batch_extract_bills.sh ``` to extract the left column of the pdf files which contain the English texts.
Then run the ```scripts/batch_clean_bills.sh``` to clean out any splitting or unaccepted regex.

### Parsing the XML for Congress

## NLP Preprocessing Pipeline

Builds two TF-IDF feature matrices from legislative bill data:
- **Title + Description** — universal coverage, already in `bills.csv`
- **Full bill text** — where extracted text files exist

Outputs saved to `data/normalized/`:

| File | Description |
|---|---|
| `bills_with_text.parquet` | bills dataframe with `full_text` column |
| `bill_ids.npy` | ordered bill IDs — row index shared by all matrices |
| `tfidf_title_desc.npz` | sparse TF-IDF on title+description |
| `tfidf_full_text.npz` | sparse TF-IDF on full bill text |
| `tfidf_vocab.json` | vocabularies for both vectorizers |

### 0. Setup

In [2]:
import json
import re
import subprocess
import zipfile
from pathlib import Path
from xml.etree import ElementTree as ET

import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer

ROOT   = Path("..").resolve()
DATA   = ROOT / "data"
NORM   = DATA / "normalized"
CA_TXT = DATA / "canada_bill_text"
US_TXT = DATA / "us_bill_text"

print("ROOT:", ROOT)

ROOT: /Users/mohsen/Documents/code/cpsc440


### 1. Generate / Load `bills.csv`

In [3]:
bills_path = NORM / "bills.csv"

if not bills_path.exists():
    print("bills.csv not found — running normalize.py ...")
    result = subprocess.run(
        ["python", str(ROOT / "scripts" / "normalize.py")],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr)
        raise RuntimeError("normalize.py failed")
else:
    print(f"bills.csv found ({bills_path.stat().st_size / 1e6:.1f} MB)")

bills = pd.read_csv(bills_path)
print(f"\nLoaded {len(bills):,} bills")
print(f"Columns: {list(bills.columns)}")
print("\nPass rate by source:")
print(bills.groupby("source")["passed"].agg(["mean", "sum", "count"]).round(3))

bills.csv found (106.1 MB)

Loaded 120,339 bills
Columns: ['bill_id', 'source', 'session', 'bill_number', 'title', 'description', 'bill_type', 'bill_type_raw', 'chamber', 'sponsor', 'party', 'introduced_date', 'status', 'passed', 'year', 'title_word_count', 'description_word_count', 'month_introduced', 'parliament_number', 'session_number', 'reinstated', 'reached_house_second_reading', 'reached_house_third_reading', 'reached_senate_third_reading', 'days_active', 'num_sponsors', 'num_history_steps', 'num_text_versions', 'num_rollcalls', 'final_yea_pct', 'has_committee']

Pass rate by source:
         mean   sum   count
source                     
canada  0.150  1065    7108
us      0.021  2387  113231


/var/folders/6m/r0nr24g501dfry13krltmw9h0000gn/T/ipykernel_3954/2694197659.py:16: DtypeWarning: Columns (0: session) have mixed types. Specify dtype option on import or set low_memory=False.
  bills = pd.read_csv(bills_path)


### 2. Load Full Bill Text

**Canada**: `data/canada_bill_text/{session}/{bill_number}_english.txt`  
**US**: XML files inside `data/us_bill_text/{congress}_{session}_congress_{type}.zip`

In [4]:
_CA_XML_LANG = "{http://www.w3.org/XML/1998/namespace}lang"
_CA_SKIP_TAGS = {"Identification", "Label", "MarginalNote"}

def _ca_xml_text(el) -> str:
    """Recursively extract English text from a Canadian bill XML element."""
    if el.tag in _CA_SKIP_TAGS:
        return ""
    # Skip French amended-text blocks
    if el.tag == "AmendedText" and el.get(_CA_XML_LANG) == "fr":
        return ""
    parts = []
    if el.tag == "Text" and el.text:
        parts.append(el.text.strip())
    for child in el:
        parts.append(_ca_xml_text(child))
        if child.tail:
            parts.append(child.tail.strip())
    return " ".join(p for p in parts if p)


def load_canada_texts(ca_txt_dir: Path) -> dict:
    """
    Returns {'{session}/{bill_number}': text}.

    Handles two source formats per session directory:
      - *_english.txt  — PDF-extracted plain text (sessions ~35-1 through ~38-1)
      - *.xml          — structured XML bills       (sessions ~39-1 onward)
    Both may coexist in transitional sessions; txt takes priority when both exist.
    """
    texts = {}
    for session_dir in sorted(ca_txt_dir.iterdir()):
        if not session_dir.is_dir():
            continue
        session = session_dir.name

        # Plain-text files (PDF-extracted English)
        for txt_file in session_dir.glob("*_english.txt"):
            bill_num = txt_file.name.replace(".pdf_english.txt", "").replace("_english.txt", "")
            texts[f"{session}/{bill_num}"] = txt_file.read_text(encoding="utf-8", errors="replace")

        # XML files — only load if no txt version already loaded
        for xml_file in session_dir.glob("*.xml"):
            bill_num = xml_file.stem          # e.g. "C-10"
            key = f"{session}/{bill_num}"
            if key in texts:                  # txt already has it
                continue
            try:
                root = ET.parse(xml_file).getroot()
                raw = _ca_xml_text(root)
                if raw:
                    texts[key] = raw
            except Exception as e:
                print(f"Warning: {xml_file.name}: {e}")

    n_txt = sum(1 for k, v in texts.items() if not v.startswith("<"))  # rough
    print(f"Loaded {len(texts):,} Canadian bill texts")
    return texts

ca_texts = load_canada_texts(CA_TXT)

Loaded 6,116 Canadian bill texts


In [5]:
_SKIP_TAGS = {
    "metadata", "dublinCore", "form", "distribution-code", "congress", "session",
    "current-chamber", "action", "action-date", "action-desc", "legis-type",
    "enum", "external-xref", "quote",
}

def _xml_text(el) -> str:
    if el.tag in _SKIP_TAGS:
        return ""
    parts = [el.text or ""]
    for child in el:
        parts.append(_xml_text(child))
        parts.append(child.tail or "")
    return " ".join(p.strip() for p in parts if p.strip())

# LegiScan bill_number prefix -> XML filename type string
_PREFIX_TO_TYPE = {
    "HB": "hr",  "SB": "s",
    "HR": "hres", "SR": "sres",
    "HJR": "hjres", "SJR": "sjres",
    "HCR": "hconres", "SCR": "sconres",
}
_TYPE_TO_PREFIX = {v: k for k, v in _PREFIX_TO_TYPE.items()}

def load_us_texts(us_txt_dir: Path) -> dict:
    """Returns {'{congress}/{bill_number}': text} e.g. {'113/HB120': '...'}"""
    texts = {}
    errors = 0
    for zip_path in sorted(us_txt_dir.glob("*.zip")):
        # 113_1_congress_hr.zip -> congress="113", xml_type="hr"
        parts = zip_path.stem.split("_")
        congress = parts[0]
        xml_type = parts[-1]
        legiscan_prefix = _TYPE_TO_PREFIX.get(xml_type)
        if legiscan_prefix is None:
            continue
        try:
            with zipfile.ZipFile(zip_path) as zf:
                for name in zf.namelist():
                    if not name.endswith(".xml"):
                        continue
                    # BILLS-113hr120ih.xml -> bill num 120
                    m = re.match(rf"BILLS-{congress}{xml_type}(\d+)", name, re.IGNORECASE)
                    if not m:
                        continue
                    key = f"{congress}/{legiscan_prefix}{int(m.group(1))}"
                    try:
                        root = ET.fromstring(zf.read(name))
                        body = root.find("./legis-body")
                        if body is not None:
                            texts[key] = _xml_text(body)
                    except Exception:
                        errors += 1
        except Exception as e:
            print(f"Warning: {zip_path.name}: {e}")
    print(f"Loaded {len(texts):,} US bill texts  ({errors} parse errors ignored)")
    return texts

us_texts = load_us_texts(US_TXT)

Loaded 88,031 US bill texts  (6 parse errors ignored)


In [6]:
bills["full_text"] = ""
mask_ca = bills["source"] == "canada"
mask_us = bills["source"] == "us"

bills.loc[mask_ca, "full_text"] = bills[mask_ca].apply(
    lambda r: ca_texts.get(f"{r['session']}/{r['bill_number']}", ""), axis=1
)
bills.loc[mask_us, "full_text"] = bills[mask_us].apply(
    lambda r: us_texts.get(f"{r['session']}/{r['bill_number']}", ""), axis=1
)

has_text = bills["full_text"].str.len() > 0
print(f"Bills with full text: {has_text.sum():,} / {len(bills):,} ({100 * has_text.mean():.1f}%)")
print("\nCoverage by source:")
print(bills.groupby("source")["full_text"].apply(lambda s: (s.str.len() > 0).mean()).round(3))

Bills with full text: 81,631 / 120,339 (67.8%)

Coverage by source:
source
canada    0.860
us        0.667
Name: full_text, dtype: float64


### 3. Text Cleaning for TF-IDF

In [7]:
import re
import nltk
import string

nltk.download("stopwords", quiet=True)
nltk.download("wordnet",   quiet=True)
nltk.download("omw-1.4",  quiet=True)

from nltk.corpus import stopwords
from nltk.stem   import WordNetLemmatizer

_LEMMATIZER = WordNetLemmatizer()
_LEG_STOP_WORDS = {
    "act", "section", "subsection", "paragraph", "clause",
    "whereas", "herein", "thereof", "hereby", "notwithstanding",
    "pursuant", "enacted", "amended",
    "mr", "mrs", "ms", "hon", "dr",   # honorifics — appear as name prefixes, not content
}
_COUNTRY_WORDS = {
    "congress", "parliament", "senate", "assented", "sanctioned",
    "elizabeth", "charles", "majesty", "minister", "secretary",
    "department", "ministry", "privy", "council", "states", "province"
}
_STOP_WORDS    = set(stopwords.words("english")) | _LEG_STOP_WORDS
_STOP_WORDS_NC = _STOP_WORDS | _COUNTRY_WORDS
_PUNCT_DIGITS  = str.maketrans("", "", string.punctuation + string.digits)

# Post-passage metadata and artifact patterns stripped before any other processing.
# Order: repeated-artifact first (uses backreference group 1); all others are phrase-level.
_LEAKAGE_RE = re.compile(
    r"(\w{3,8}?)\1{4,}"                                  # run-together OCR artifact (cidcidcid...)
    r"|\bassented\b[^\n.;]*"                              # "ASSENTED TO 12th MAY, 1994"
    r"|\bstatutes?\s+of\s+canada\b"                       # "STATUTES OF CANADA"
    r"|\blois\s+du\s+canada\b"                            # French equivalent
    r"|\bunited\s+states\b"                               # "United States" (phrase — both variants)
    r"|\bcanada\b|\bcanadian\b|\bamerican\b"              # country names removed in both variants
    r"|\bchapter\s+\d+\b"                                 # "CHAPTER 13"
    r"|\bc\.\s*\d+\b"                                     # "C. 13"
    r"|\b\d+\s*[-\u2013]\s*\d+\s+elizabeth\s+ii\b"       # "42-43 Elizabeth II"
    r"|\belizabeth\s+ii\b"                                # bare "Elizabeth II"
    r"|\b(?:first|second|third)\s+reading\b"              # procedural stage markers (full phrase)
    r"|\breading\b"                                       # standalone "reading"
    r"|\b\w[\w-]*\s+(?:session|parliament)\b"             # "First Session", "Thirty-fifth Parliament"
    r"|\b(?:january|february|march|april|may|june|july"
    r"|august|september|october|november|december)\b"     # enactment date month names
    r"|\b(?:her|his)\s+majesty\b"                         # "Her Majesty"
    r"|by\s+and\s+with\s+the\s+advice\s+and\s+consent",  # royal assent boilerplate
    re.IGNORECASE,
)

def _strip_leakage(text: str) -> str:
    return _LEAKAGE_RE.sub(" ", text)

def _tokenize(text: str, stop_words: set) -> str:
    if not isinstance(text, str) or not text.strip():
        return ""
    text = _strip_leakage(text)                # leakage removal before any other step
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = text.translate(_PUNCT_DIGITS)
    return " ".join(
        _LEMMATIZER.lemmatize(w)
        for w in text.split()
        if w not in stop_words and 1 < len(w) <= 25  # drop honorifics AND run-together artifacts
    )

def clean_bill_text(text: str) -> str:
    """Strip leakage + stop words + punct/digits, then lemmatize."""
    return _tokenize(text, _STOP_WORDS)

def clean_bill_text_nc(text: str) -> str:
    """Same as clean_bill_text but also strips country-signalling words."""
    return _tokenize(text, _STOP_WORDS_NC)

def clean_text(text) -> str:
    """Lightweight cleaner for title/description (kept for downstream TF-IDF cells)."""
    if not isinstance(text, str) or not text:
        return ""
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


_clean_path = NORM / "bills_clean.csv"
_needs_clean = True

if _clean_path.exists():
    _needs_clean = False
    bills = pd.read_csv(_clean_path, low_memory=False)
    print(f"Loaded bills_clean.csv — {len(bills):,} bills  ({_clean_path.stat().st_size / 1e6:.1f} MB)")

if _needs_clean:
    bills["text_clean"]    = bills["full_text"].fillna("").apply(clean_bill_text)
    bills["text_clean_nc"] = bills["full_text"].fillna("").apply(clean_bill_text_nc)
    if "title_clean" not in bills.columns:
        bills["title_clean"]       = bills["title"].fillna("").apply(clean_text)
        bills["description_clean"] = bills["description"].fillna("").apply(clean_text)
        bills["title_desc"]        = bills["title_clean"] + " " + bills["description_clean"]
        bills["full_text_clean"]   = bills["full_text"].fillna("").apply(clean_text)
    bills.to_csv(_clean_path, index=False)
    print(f"Saved bills_clean.csv  ({len(bills):,} rows, {bills['text_clean'].str.len().gt(0).sum():,} with cleaned text)")

# Sanity check — 3 Canadian bills where raw text contained 'assented'
_ca_assented = bills[
    (bills["source"] == "canada") &
    (bills["full_text"].str.contains("assented", case=False, na=False))
].head(3)
print(f"\nSanity check — {len(_ca_assented)} Canadian bills with 'assented' in raw text:")
for i, (_, row) in enumerate(_ca_assented.iterrows(), 1):
    print(f"\n--- Bill {i}: {row['bill_number']} (passed={row['passed']}) ---")
    print(f"  raw       : {row['full_text'][:300]!r}")
    print(f"  text_clean: {row['text_clean'][:300]!r}")

Loaded bills_clean.csv — 120,339 bills  (3613.8 MB)

Sanity check — 3 Canadian bills with 'assented' in raw text:

--- Bill 1: C-9 (passed=1) ---
  raw       : 'r.s., c. 1 (5th income tax act\nsupp.); 1991,\ncc. 47, 49;\n1992, cc. 1,\n24, 27, 29, 48;\n1993, cc. 24, 27\n1. (1) the portion of the definition\n‘‘qualifying debt obligation’’ in subsection 15.1(3) of the income tax act before\nparagraph (a) is replaced by the following:\n‘‘qualifying ‘‘qualifying debt obl'
  text_clean: 'r th income tax supp cc cc cc portion definition ‘‘qualifying debt obligation’’ income tax replaced following ‘‘qualifying ‘‘qualifying debt obligation’’ corporation debt particular time mean obligation obligation’’ «créance bond debenture bill note mortgage admissible» similar obligation issued app'

--- Bill 2: C-10 (passed=1) ---
  raw       : 'short title\nshort title 1. this act may be cited as the west coast\nports operations act, 1994.\ninterpretation\ndefinitions 2. (1) in this act,\n‘‘collective ‘‘c

### 4. TF-IDF Vectorization

In [8]:
# Title + Description (universal coverage)
tfidf_td = TfidfVectorizer(
    max_features=10_000,
    min_df=5,
    max_df=0.95,
    ngram_range=(1, 2),
    sublinear_tf=True,
    stop_words="english",
)
X_td = tfidf_td.fit_transform(bills["title_desc"])
print(f"Title+Desc TF-IDF: {X_td.shape}  |  nnz={X_td.nnz:,}")

vocab_td = np.array(tfidf_td.get_feature_names_out())
top20_td = vocab_td[np.asarray(X_td.mean(axis=0)).flatten().argsort()[-20:][::-1]]
print("Top 20 terms:", list(top20_td))

Title+Desc TF-IDF: (120339, 10000)  |  nnz=5,745,197
Top 20 terms: ['act', 'purposes', 'amend', 'states', 'act amend', 'united', 'united states', 'national', 'federal', 'health', 'certain', 'program', 'provide', 'code', 'amends', 'secretary', 'state', 'department', 'security', 'resolution']


### 5. TF-IDF + Logistic Regression on Full Bill Text

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, average_precision_score
import pandas as pd

# Temporal cutoffs — must match combined_model.ipynb for a fair comparison
US_CUTOFF = 2022
CA_CUTOFF = 2020

VARIANTS = {
    "standard":   "text_clean",
    "no-country": "text_clean_nc",
}

results = {}   # variant -> {"overall": ap, "canada": ap, "us": ap}
models  = {}   # variant -> {"tfidf": ..., "lr": ...}

for label, col in VARIANTS.items():
    print(f"\n{'='*60}")
    print(f"  Variant: {label}  (column: {col})")
    print(f"{'='*60}")

    df_text = bills[bills[col].str.len() > 0].dropna(subset=["year"]).copy()
    df_text["year"] = df_text["year"].astype(int)
    print(f"Working set: {len(df_text):,} bills  (pass rate: {df_text['passed'].mean():.3f})")

    us_data = df_text[df_text["source"] == "us"]
    ca_data = df_text[df_text["source"] == "canada"]

    train_df = pd.concat([
        us_data[us_data["year"] <= US_CUTOFF],
        ca_data[ca_data["year"] <= CA_CUTOFF],
    ])
    test_df = pd.concat([
        us_data[us_data["year"] > US_CUTOFF],
        ca_data[ca_data["year"] > CA_CUTOFF],
    ])

    X_raw      = train_df[col]
    X_test_raw = test_df[col]
    y_train    = train_df["passed"]
    y_test     = test_df["passed"]
    src_train  = train_df["source"]
    src_test   = test_df["source"]

    print(f"Train: {len(train_df):,}  Test: {len(test_df):,}")

    tfidf = TfidfVectorizer(
        max_features=20_000, min_df=5, max_df=0.95,
        ngram_range=(1, 2), sublinear_tf=True,
    )
    X_train = tfidf.fit_transform(X_raw)
    X_test  = tfidf.transform(X_test_raw)
    print(f"TF-IDF: train={X_train.shape}  test={X_test.shape}")

    lr = LogisticRegression(C=1.0, class_weight="balanced", max_iter=1000, random_state=42)
    lr.fit(X_train, y_train)
    models[label] = {"tfidf": tfidf, "lr": lr}

    y_pred = lr.predict(X_test)
    y_prob = lr.predict_proba(X_test)[:, 1]

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=["failed", "passed"]))

    ap_overall = average_precision_score(y_test, y_prob)
    print(f"Overall PR-AUC: {ap_overall:.4f}")

    per_src = {}
    for src in ("canada", "us"):
        mask = (src_test == src).values
        if mask.sum() == 0:
            continue
        ap = average_precision_score(y_test.values[mask], y_prob[mask])
        per_src[src] = ap
        print(f"{src.upper()} PR-AUC ({mask.sum():,} bills): {ap:.4f}")

    results[label] = {"overall": ap_overall, **per_src}

    vocab = np.array(tfidf.get_feature_names_out())
    coef  = lr.coef_[0]
    print(f"\nTop 10 → passed : {list(vocab[coef.argsort()[-10:][::-1]])}")
    print(f"Top 10 → failed : {list(vocab[coef.argsort()[:10]])}")

# Side-by-side summary
print(f"\n{'='*60}")
print(f"  Summary")
print(f"{'='*60}")
header = f"{'':25s}" + "".join(f"{v:>14s}" for v in VARIANTS)
print(header)
for metric, label in [("overall", "Overall PR-AUC"), ("canada", "Canada  PR-AUC"), ("us", "US      PR-AUC")]:
    row = f"{label:25s}" + "".join(f"{results[v].get(metric, float('nan')):14.4f}" for v in VARIANTS)
    print(row)

# Store named references for downstream cells
tfidf_std, lr_std = models["standard"]["tfidf"],   models["standard"]["lr"]
tfidf_nc,  lr_nc  = models["no-country"]["tfidf"], models["no-country"]["lr"]



  Variant: standard  (column: text_clean)
Working set: 80,841 bills  (pass rate: 0.032)
Train: 63,992  Test: 16,849
TF-IDF: train=(63992, 20000)  test=(16849, 20000)

Classification Report:
              precision    recall  f1-score   support

      failed       0.99      0.92      0.96     16577
      passed       0.13      0.71      0.21       272

    accuracy                           0.92     16849
   macro avg       0.56      0.81      0.58     16849
weighted avg       0.98      0.92      0.94     16849

Overall PR-AUC: 0.2805
CANADA PR-AUC (663 bills): 0.4768
US PR-AUC (16,186 bills): 0.1388

Top 10 → passed : ['display', 'province', 'format', 'encourage', 'later year', 'senate', 'replaced', 'accordance', 'airport', 'extension']
Top 10 → failed : ['existing text', 'percent', 'relevant portion', 'following existing', 'alien', 'receiving', 'establishment', 'trade', 'taxable year', 'later one']

  Variant: no-country  (column: text_clean_nc)
Working set: 80,841 bills  (pass rate:

### 6. Cross-National Transfer Experiments (no-country variant)

Train on one country, test on the other — measures whether legislative language generalises across systems.

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, average_precision_score, roc_auc_score

col = "text_clean_nc"
df_nc = bills[bills[col].str.len() > 0].dropna(subset=["year"]).copy()
df_nc["year"] = df_nc["year"].astype(int)

df_us = df_nc[df_nc["source"] == "us"].reset_index(drop=True)
df_ca = df_nc[df_nc["source"] == "canada"].reset_index(drop=True)

print(f"US: {len(df_us):,} bills  (pass rate {df_us['passed'].mean():.3f})")
print(f"CA: {len(df_ca):,} bills  (pass rate {df_ca['passed'].mean():.3f})")


def _fit_eval(train_df, test_df, experiment_label):
    tfidf = TfidfVectorizer(
        max_features=20_000, min_df=5, max_df=0.95,
        ngram_range=(1, 2), sublinear_tf=True,
    )
    lr = LogisticRegression(C=1.0, class_weight="balanced", max_iter=1000, random_state=42)

    X_tr = tfidf.fit_transform(train_df[col])
    X_te = tfidf.transform(test_df[col])
    lr.fit(X_tr, train_df["passed"])

    y_te   = test_df["passed"].values
    y_pred = lr.predict(X_te)
    y_prob = lr.predict_proba(X_te)[:, 1]

    pr  = average_precision_score(y_te, y_prob)
    roc = roc_auc_score(y_te, y_prob)

    print(f"\n{'='*60}")
    print(f"  {experiment_label}")
    print(f"{'='*60}")
    print(f"Train: {len(train_df):,} bills  |  Test: {len(test_df):,} bills")
    print(classification_report(y_te, y_pred, target_names=["failed", "passed"]))
    print(f"PR-AUC:  {pr:.4f}   ROC-AUC: {roc:.4f}")

    vocab = np.array(tfidf.get_feature_names_out())
    coef  = lr.coef_[0]
    print(f"Top 10 → passed : {list(vocab[coef.argsort()[-10:][::-1]])}")
    print(f"Top 10 → failed : {list(vocab[coef.argsort()[:10]])}")

    return pr, roc


# In-distribution baselines — temporal split per country
us_tr = df_us[df_us["year"] <= US_CUTOFF]
us_te = df_us[df_us["year"] >  US_CUTOFF]
ca_tr = df_ca[df_ca["year"] <= CA_CUTOFF]
ca_te = df_ca[df_ca["year"] >  CA_CUTOFF]

pr_us_id,  roc_us_id  = _fit_eval(us_tr, us_te, "In-distribution: Train US → Test US")
pr_ca_id,  roc_ca_id  = _fit_eval(ca_tr, ca_te, "In-distribution: Train CA → Test CA")

# Transfer: train on all of one country, test on all of the other
pr_us_ca, roc_us_ca = _fit_eval(df_us, df_ca, "Transfer Exp 1: Train US → Test CA")
pr_ca_us, roc_ca_us = _fit_eval(df_ca, df_us, "Transfer Exp 2: Train CA → Test US")

# Summary table
transfer_summary = [
    ("US in-distribution",  "US", pr_us_id,  roc_us_id),
    ("CA → US  (transfer)", "US", pr_ca_us,  roc_ca_us),
    ("CA in-distribution",  "CA", pr_ca_id,  roc_ca_id),
    ("US → CA  (transfer)", "CA", pr_us_ca,  roc_us_ca),
]
print(f"\n{'='*60}")
print(f"  Transfer Summary  (text_clean_nc)")
print(f"{'='*60}")
print(f"{'Experiment':<35s} {'Test':>4s} {'PR-AUC':>8s} {'ROC-AUC':>9s}")
print("-" * 60)
for label, src, pr, roc in transfer_summary:
    print(f"{label:<35s} {src:>4s} {pr:8.4f} {roc:9.4f}")


US: 75,515 bills  (pass rate 0.023)
CA: 5,326 bills  (pass rate 0.168)

  In-distribution: Train US → Test US
Train: 59,329 bills  |  Test: 16,186 bills
              precision    recall  f1-score   support

      failed       1.00      0.92      0.96     16034
      passed       0.07      0.61      0.12       152

    accuracy                           0.92     16186
   macro avg       0.53      0.77      0.54     16186
weighted avg       0.99      0.92      0.95     16186

PR-AUC:  0.1372   ROC-AUC: 0.8595
Top 10 → passed : ['display', 'airport', 'encourage', 'extension', 'later year', 'independence', 'disabled', 'stat', 'veteran', 'memorial']
Top 10 → failed : ['percent', 'least', 'alien', 'qualified', 'taxable year', 'receiving', 'subparagraphs', 'one year', 'establishment', 'taxable']

  In-distribution: Train CA → Test CA
Train: 4,663 bills  |  Test: 663 bills
              precision    recall  f1-score   support

      failed       0.92      0.76      0.83       543
      passed

### 7. Fusion Model: TF-IDF Text + Structured Metadata

Combines two feature branches fed into a single Logistic Regression:
- **TF-IDF branch** — same 20k-feature no-country cleaned bill text as §5
- **Metadata branch** — one-hot categorical (`chamber`, `bill_type`, `party`) + scaled numeric (`year`, `month_introduced`, `title_word_count`, `description_word_count`, `num_sponsors`)

Excluded as leakage: `final_yea_pct`, `num_rollcalls`, `days_active`, `num_history_steps`, `reached_*_reading`, `status`.

In [11]:
import scipy.sparse as sp
from sklearn.preprocessing import OneHotEncoder, StandardScaler

CAT_COLS = ["chamber", "bill_type", "party"]
NUM_COLS = ["year", "month_introduced", "title_word_count",
            "description_word_count", "num_sponsors"]

col = "text_clean_nc"
df_fuse = bills[bills[col].str.len() > 0].dropna(subset=["year"]).copy().reset_index(drop=True)
df_fuse["year"] = df_fuse["year"].astype(int)
df_fuse[CAT_COLS] = df_fuse[CAT_COLS].fillna("unknown")
df_fuse[NUM_COLS] = df_fuse[NUM_COLS].fillna(0)
print(f"Working set: {len(df_fuse):,} bills  (pass rate: {df_fuse['passed'].mean():.3f})")

# ── Temporal split per country ────────────────────────────────────────────────
us_fuse = df_fuse[df_fuse["source"] == "us"]
ca_fuse = df_fuse[df_fuse["source"] == "canada"]

train_df = pd.concat([
    us_fuse[us_fuse["year"] <= US_CUTOFF],
    ca_fuse[ca_fuse["year"] <= CA_CUTOFF],
])
test_df = pd.concat([
    us_fuse[us_fuse["year"] > US_CUTOFF],
    ca_fuse[ca_fuse["year"] > CA_CUTOFF],
])
print(f"Train: {len(train_df):,}  Test: {len(test_df):,}")

# ── TF-IDF branch ────────────────────────────────────────────
tfidf_fuse = TfidfVectorizer(
    max_features=20_000, min_df=5, max_df=0.95,
    ngram_range=(1, 2), sublinear_tf=True,
)
X_text_train = tfidf_fuse.fit_transform(train_df[col])
X_text_test  = tfidf_fuse.transform(test_df[col])

# ── Metadata branch ──────────────────────────────────────────
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
X_cat_train = ohe.fit_transform(train_df[CAT_COLS])
X_cat_test  = ohe.transform(test_df[CAT_COLS])

scaler = StandardScaler()
X_num_train = sp.csr_matrix(scaler.fit_transform(train_df[NUM_COLS]))
X_num_test  = sp.csr_matrix(scaler.transform(test_df[NUM_COLS]))

# ── Fuse: hstack all branches ───────────────────────────────────
X_train_fused = sp.hstack([X_text_train, X_cat_train, X_num_train], format="csr")
X_test_fused  = sp.hstack([X_text_test,  X_cat_test,  X_num_test],  format="csr")
print(f"Fused matrix — train: {X_train_fused.shape}  test: {X_test_fused.shape}")

# ── Logistic Regression on fused features ──────────────────────────
lr_fused = LogisticRegression(C=1.0, class_weight="balanced", max_iter=1000, random_state=42)
lr_fused.fit(X_train_fused, train_df["passed"])

y_test_f   = test_df["passed"].values
y_pred_f   = lr_fused.predict(X_test_fused)
y_prob_f   = lr_fused.predict_proba(X_test_fused)[:, 1]
src_test_f = test_df["source"].values

print("\nClassification Report:")
print(classification_report(y_test_f, y_pred_f, target_names=["failed", "passed"]))

ap_fused = average_precision_score(y_test_f, y_prob_f)
print(f"Overall PR-AUC: {ap_fused:.4f}")

ap_fused_per = {}
for src in ("canada", "us"):
    mask = src_test_f == src
    if mask.sum() == 0:
        continue
    ap = average_precision_score(y_test_f[mask], y_prob_f[mask])
    ap_fused_per[src] = ap
    print(f"{src.upper()} PR-AUC ({mask.sum():,} bills): {ap:.4f}")

# ── Comparison table ─────────────────────────────────────────────
baseline = results["no-country"]
print(f"\n{'─'*55}")
print(f"{'Model':<32s} {'Overall':>9s} {'Canada':>9s} {'US':>6s}")
print(f"{'─'*55}")
print(f"{'TF-IDF only (§5)':32s} {baseline['overall']:>9.4f} {baseline['canada']:>9.4f} {baseline['us']:>6.4f}")
print(f"{'TF-IDF + metadata (fusion)':32s} {ap_fused:>9.4f} {ap_fused_per.get('canada', float('nan')):>9.4f} {ap_fused_per.get('us', float('nan')):>6.4f}")


Working set: 80,841 bills  (pass rate: 0.032)
Train: 63,992  Test: 16,849
Fused matrix — train: (63992, 20016)  test: (16849, 20016)

Classification Report:
              precision    recall  f1-score   support

      failed       0.99      0.95      0.97     16577
      passed       0.17      0.66      0.27       272

    accuracy                           0.94     16849
   macro avg       0.58      0.80      0.62     16849
weighted avg       0.98      0.94      0.96     16849

Overall PR-AUC: 0.2817
CANADA PR-AUC (663 bills): 0.4614
US PR-AUC (16,186 bills): 0.1448

───────────────────────────────────────────────────────
Model                              Overall    Canada     US
───────────────────────────────────────────────────────
TF-IDF only (§5)                    0.2813    0.4825 0.1408
TF-IDF + metadata (fusion)          0.2817    0.4614 0.1448


### 8. Cross-National Transfer Experiments: Fusion Model

Same train-on-one-country / test-on-the-other design as §6, but using the full fusion pipeline (TF-IDF + one-hot metadata + scaled numerics). All transformers are fit on the training country only.

In [12]:
from sklearn.metrics import roc_auc_score

col = "text_clean_nc"
df_nc = bills[bills[col].str.len() > 0].dropna(subset=["year"]).copy()
df_nc["year"] = df_nc["year"].astype(int)

df_us = df_nc[df_nc["source"] == "us"].reset_index(drop=True)
df_ca = df_nc[df_nc["source"] == "canada"].reset_index(drop=True)

for df in (df_us, df_ca):
    df[CAT_COLS] = df[CAT_COLS].fillna("unknown")
    df[NUM_COLS] = df[NUM_COLS].fillna(0)


def _fit_eval_fused(train_df, test_df, experiment_label):
    # TF-IDF branch
    tfidf = TfidfVectorizer(
        max_features=20_000, min_df=5, max_df=0.95,
        ngram_range=(1, 2), sublinear_tf=True,
    )
    X_text_tr = tfidf.fit_transform(train_df[col])
    X_text_te = tfidf.transform(test_df[col])

    # Metadata branch
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    X_cat_tr = ohe.fit_transform(train_df[CAT_COLS])
    X_cat_te = ohe.transform(test_df[CAT_COLS])

    scaler = StandardScaler()
    X_num_tr = sp.csr_matrix(scaler.fit_transform(train_df[NUM_COLS]))
    X_num_te = sp.csr_matrix(scaler.transform(test_df[NUM_COLS]))

    X_tr = sp.hstack([X_text_tr, X_cat_tr, X_num_tr], format="csr")
    X_te = sp.hstack([X_text_te, X_cat_te, X_num_te], format="csr")

    lr = LogisticRegression(C=1.0, class_weight="balanced", max_iter=1000, random_state=42)
    lr.fit(X_tr, train_df["passed"])

    y_te   = test_df["passed"].values
    y_pred = lr.predict(X_te)
    y_prob = lr.predict_proba(X_te)[:, 1]

    pr  = average_precision_score(y_te, y_prob)
    roc = roc_auc_score(y_te, y_prob)

    print(f"\n{'='*60}")
    print(f"  {experiment_label}")
    print(f"{'='*60}")
    print(f"Train: {len(train_df):,} bills  |  Test: {len(test_df):,} bills")
    print(classification_report(y_te, y_pred, target_names=["failed", "passed"]))
    print(f"PR-AUC:  {pr:.4f}   ROC-AUC: {roc:.4f}")
    return pr, roc


# In-distribution baselines — temporal split per country
us_tr = df_us[df_us["year"] <= US_CUTOFF]
us_te = df_us[df_us["year"] >  US_CUTOFF]
ca_tr = df_ca[df_ca["year"] <= CA_CUTOFF]
ca_te = df_ca[df_ca["year"] >  CA_CUTOFF]

pr_us_id_f,  roc_us_id_f  = _fit_eval_fused(us_tr, us_te, "In-distribution: Train US → Test US")
pr_ca_id_f,  roc_ca_id_f  = _fit_eval_fused(ca_tr, ca_te, "In-distribution: Train CA → Test CA")

# Transfer
pr_us_ca_f, roc_us_ca_f = _fit_eval_fused(df_us, df_ca, "Transfer Exp 1: Train US → Test CA")
pr_ca_us_f, roc_ca_us_f = _fit_eval_fused(df_ca, df_us, "Transfer Exp 2: Train CA → Test US")

# Summary vs §6 text-only results (from cell above)
print(f"\n{'='*68}")
print(f"  Transfer Summary: TF-IDF only (§6) vs Fusion (§8)")
print(f"{'='*68}")
print(f"{'Experiment':<35s} {'Test':>4s} {'TF-IDF PR':>10s} {'Fusion PR':>10s} {'TF-IDF ROC':>11s} {'Fusion ROC':>11s}")
print("-" * 68)
rows = [
    ("US in-distribution",  "US", pr_us_id,   roc_us_id,   pr_us_id_f,   roc_us_id_f),
    ("CA → US  (transfer)", "US", pr_ca_us,   roc_ca_us,   pr_ca_us_f,   roc_ca_us_f),
    ("CA in-distribution",  "CA", pr_ca_id,   roc_ca_id,   pr_ca_id_f,   roc_ca_id_f),
    ("US → CA  (transfer)", "CA", pr_us_ca,   roc_us_ca,   pr_us_ca_f,   roc_us_ca_f),
]
for label, src, pr_t, roc_t, pr_f, roc_f in rows:
    print(f"{label:<35s} {src:>4s} {pr_t:>10.4f} {pr_f:>10.4f} {roc_t:>11.4f} {roc_f:>11.4f}")



  In-distribution: Train US → Test US
Train: 59,329 bills  |  Test: 16,186 bills
              precision    recall  f1-score   support

      failed       1.00      0.95      0.97     16034
      passed       0.10      0.55      0.17       152

    accuracy                           0.95     16186
   macro avg       0.55      0.75      0.57     16186
weighted avg       0.99      0.95      0.97     16186

PR-AUC:  0.1422   ROC-AUC: 0.8777

  In-distribution: Train CA → Test CA
Train: 4,663 bills  |  Test: 663 bills
              precision    recall  f1-score   support

      failed       0.91      0.86      0.88       543
      passed       0.49      0.59      0.53       120

    accuracy                           0.81       663
   macro avg       0.70      0.73      0.71       663
weighted avg       0.83      0.81      0.82       663

PR-AUC:  0.5326   ROC-AUC: 0.8084

  Transfer Exp 1: Train US → Test CA
Train: 75,515 bills  |  Test: 5,326 bills
              precision    recall  f1-